## Silver to Gold Transformation Notebook
### Healthcare Project Group C — Gold Layer
**7 Business Requirements**

| BR | Gold Table | Silver Sources | Grain |
|----|-----------|----------------|-------|
| BR-01 | `gold_patient_profile` | patients, encounters, diagnoses, insurance | 1 row / patient |
| BR-02 | `gold_encounter_analytics` | encounters, patients, facilities, providers | 1 row / encounter |
| BR-03 | `gold_billing_summary` | billing, encounters, insurance, patients | 1 row / billing |
| BR-04 | `gold_diagnosis_summary` | diagnoses, patients, encounters | 1 row / patient / year |
| BR-05 | `gold_medication_insights` | medications, providers, patients | 1 row / prescriber / drug |
| BR-06 | `gold_lab_analytics` | lab_results, encounters, patients | 1 row / patient / test |
| BR-07 | `gold_provider_performance` | providers, encounters, billing, diagnoses | 1 row / provider |


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder.appName("Silver_to_Gold_Healthcare_GroupC").getOrCreate()
print("Spark version:", spark.version)

In [0]:
# ── Catalog / Schema Configuration ──────────────────────────────────────────

silver_path = "abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/"
gold_path = "abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/"


patients_silver_df = spark.read.format("delta").load(silver_path + "patients")
providers_silver_df = spark.read.format("delta").load(silver_path + "providers")
encounters_silver_df = spark.read.format("delta").load(silver_path + "encounters")
vitals_silver_df = spark.read.format("delta").load(silver_path + "vitals")
insurance_silver_df = spark.read.format("delta").load(silver_path + "insurance")
lab_results_silver_df = spark.read.format("delta").load(silver_path + "lab_results")
medications_silver_df = spark.read.format("delta").load(silver_path + "medications")
billing_silver_df = spark.read.format("delta").load(silver_path + "billing")
diagnoses_silver_df = spark.read.format("delta").load(silver_path + "diagnoses")
facilities_silver_df = spark.read.format("delta").load(silver_path + "facilities")


---
### BR-01 — Patient Demographics & Risk Profile
**Gold Table:** `gold_patient_profile`  
**Grain:** 1 row per `patient_id`  
**Sources:** `patients_silver_df`, `encounters_silver_df`, `diagnoses_silver_df`, `insurance_silver_df`


In [0]:
# ── BR-01 Step 1: Encounter aggregations per patient ─────────────────────────
enc_agg = (
    encounters_silver_df
    .groupBy("patient_id")
    .agg(
        F.countDistinct("encounter_id")
         .alias("total_visits"),
        F.countDistinct(
            F.when(F.col("encounter_type_std") == "Inpatient", F.col("encounter_id"))
        ).alias("inpatient_visits"),
        F.sum(F.coalesce(F.col("readmission_flag_std"), F.lit(0)))
         .alias("readmission_count"),
        F.max(F.to_date(F.col("admission_ts")))
         .alias("last_visit_date")
    )
)
#display(enc_agg.limit(10))



In [0]:
# ── BR-01 Step 2: Diagnosis aggregations per patient ─────────────────────────
dx_agg = (
    diagnoses_silver_df
    .groupBy("patient_id")
    .agg(
        F.count(
            F.when(F.col("chronic_flag_std") == 1, F.col("diagnosis_id"))
        ).alias("chronic_diagnosis_count"),
        F.round(F.avg("severity_score"), 2)
         .alias("avg_severity_score"),
        F.max("is_mental_health_dx")
         .alias("has_mental_health_dx"),
        F.collect_set("icd10_code_std")
         .alias("diagnosis_list")
    )
)

# Fill has_mental_health_dx default 0 where no diagnoses exist
dx_agg = dx_agg.withColumn(
    "has_mental_health_dx",
    F.coalesce(F.col("has_mental_health_dx"), F.lit(0))
)

#display(dx_agg.limit(10))

In [0]:
# ── BR-01 Step 3: Insurance coverage tier per patient (highest active tier) ──
tier_order = (
    F.when(F.col("coverage_tier") == "Premium",  4)
     .when(F.col("coverage_tier") == "Enhanced", 3)
     .when(F.col("coverage_tier") == "Standard", 2)
     .when(F.col("coverage_tier") == "Basic",    1)
     .otherwise(0)
)

ins_ranked = (
    insurance_silver_df
    .filter(F.col("is_active_flag") == 1)
    .withColumn("tier_rank", tier_order)
)

ins_agg = (
    ins_ranked
    .groupBy("patient_id")
    .agg(
        F.first(
            F.col("coverage_tier"),
            ignorenulls=True
        ).alias("insurance_coverage_tier")
    )
)
#display(ins_agg.limit(10))

In [0]:
# ── BR-01 Step 4: Build patient location derived column, then join all aggs ──
patients_base = (
    patients_silver_df
    .withColumn(
        "patient_location",
        F.when(
            F.col("city_clean").isNull() & F.col("state_clean").isNull(),
            F.lit(None)
        ).otherwise(
            F.concat_ws(", ", F.col("city_clean"), F.col("state_clean"))
        )
    )
    .select(
        "patient_id", "full_name", "age_years", "age_group",
        "gender_std", "blood_group_std", "patient_location",
        "is_active_flag", "dq_score"
    )
)

gold_patient_profile = (
    patients_base
    .join(enc_agg,  "patient_id", "left")
    .join(dx_agg,   "patient_id", "left")
    .join(ins_agg,  "patient_id", "left")
    .withColumn("total_visits",
        F.coalesce(F.col("total_visits"), F.lit(0)))
    .withColumn("inpatient_visits",
        F.coalesce(F.col("inpatient_visits"), F.lit(0)))
    .withColumn("readmission_count",
        F.coalesce(F.col("readmission_count"), F.lit(0)))
    .withColumn("chronic_diagnosis_count",
        F.coalesce(F.col("chronic_diagnosis_count"), F.lit(0)))
    .withColumn("has_mental_health_dx",
        F.coalesce(F.col("has_mental_health_dx"), F.lit(0)))
    .withColumn("diagnosis_list",
        F.coalesce(F.col("diagnosis_list"), F.array()))
    .withColumn("insurance_coverage_tier",
        F.coalesce(F.col("insurance_coverage_tier"), F.lit("None")))
    .select(
        "patient_id", "full_name", "age_years", "age_group",
        "gender_std", "blood_group_std", "patient_location",
        "insurance_coverage_tier",
        "total_visits", "inpatient_visits", "readmission_count",
        "last_visit_date", "chronic_diagnosis_count",
        "avg_severity_score", "has_mental_health_dx", "diagnosis_list",
        "is_active_flag", "dq_score"
    )
)

print(f"[BR-01] gold_patient_profile row count : {gold_patient_profile.count():,}")
#display(gold_patient_profile.limit(10))

In [0]:
# ── BR-01 Write gold_patient_profile ─────────────────────────────────────────
(
    gold_patient_profile
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/Gold_Patient_Profile')
)
print("[BR-01] gold_patient_profile written successfully.")

---
###  BR-02 — Encounter & Length-of-Stay Analytics
**Gold Table:** `gold_encounter_analytics`  
**Grain:** 1 row per `encounter_id`  
**Sources:** `encounters_silver_df`, `patients_silver_df`, `facilities_silver_df`, `providers_silver_df`


In [0]:
# ── BR-02 Step 1: Build encounter_period derived column ──────────────────────
enc_base = (
    encounters_silver_df
    .withColumn(
        "encounter_period",
        F.when(
            F.col("encounter_year").isNull() | F.col("encounter_month").isNull(),
            F.lit(None)
        ).otherwise(
            F.concat(
                F.col("encounter_year").cast("string"),
                F.lit("-"),
                F.lpad(F.col("encounter_month").cast("string"), 2, "0")
            )
        )
    )
    .select(
        "encounter_id", "patient_id", "admission_ts", "discharge_ts",
        "los_hours_final", "los_days", "encounter_type_std", "department_std",
        "readmission_flag_std", "triage_level_num", "discharge_disp_std",
        "total_charges_valid", "encounter_period", "dq_encounter_flag",
        "facility_id", "provider_id"
    )
)
#display(enc_base.limit(10))

In [0]:
# ── BR-02 Step 2: Window function – avg LOS by department ────────────────────
dept_los_window = Window.partitionBy("department_std")

enc_with_dept_avg = enc_base.withColumn(
    "avg_los_by_dept",
    F.round(F.avg("los_days").over(dept_los_window), 2)
)
#display(enc_with_dept_avg.limit(10))

In [0]:
# ── BR-02 Step 3: Join patients, facilities, providers ───────────────────────
pat_lookup = (
    patients_silver_df
    .select(
        F.col("patient_id"),
        F.col("age_group").alias("patient_age_group"),
        F.col("gender_std").alias("patient_gender")
    )
)

fac_lookup = (
    facilities_silver_df
    .select(
        F.col("facility_id"),
        F.col("hospital_size_category").alias("hospital_size"),
        F.col("is_nabh_accredited")
    )
)

prov_lookup = (
    providers_silver_df
    .select(
        F.col("provider_id"),
        F.col("specialty_std").alias("treating_specialty")
    )
)

gold_encounter_analytics = (
    enc_with_dept_avg
    .join(pat_lookup,  "patient_id",  "left")
    .join(fac_lookup,  "facility_id", "left")
    .join(prov_lookup, "provider_id", "left")
    .withColumn("patient_age_group",
        F.coalesce(F.col("patient_age_group"), F.lit("Unknown")))
    .withColumn("patient_gender",
        F.coalesce(F.col("patient_gender"),    F.lit("Unknown")))
    .withColumn("hospital_size",
        F.coalesce(F.col("hospital_size"),     F.lit("Unknown")))
    .withColumn("is_nabh_facility",
        F.coalesce(F.col("is_nabh_accredited"), F.lit(0)))
    .withColumn("treating_specialty",
        F.coalesce(F.col("treating_specialty"), F.lit("General")))
    .withColumn("readmission_flag",
        F.coalesce(F.col("readmission_flag_std"), F.lit(0)))
    .select(
        "encounter_id", "patient_id", "admission_ts", "discharge_ts",
        "los_hours_final", "los_days", "avg_los_by_dept",
        "encounter_type_std", "department_std", "encounter_period",
        "readmission_flag", "triage_level_num", "discharge_disp_std",
        "total_charges_valid", "patient_age_group", "patient_gender",
        "hospital_size", "is_nabh_facility", "treating_specialty",
        "dq_encounter_flag"
    )
)

print(f"[BR-02] gold_encounter_analytics row count : {gold_encounter_analytics.count():,}")
display(gold_encounter_analytics.limit(10))

In [0]:
# ── BR-02 Write gold_encounter_analytics ─────────────────────────────────────
(
    gold_encounter_analytics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("encounter_period")
    .save(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/Gold_Encounter_Analytics')
)
print("[BR-02] gold_encounter_analytics written successfully.")

---
###  BR-03 — Revenue & Billing Intelligence
**Gold Table:** `gold_billing_summary`  
**Grain:** 1 row per `billing_id`  
**Sources:** `billing_silver_df`, `encounters_silver_df`, `insurance_silver_df`, `patients_silver_df`


In [0]:
# ── BR-03 Step 1: Department-level AR window aggregation ─────────────────────
enc_dept = (
    encounters_silver_df
    .select("encounter_id", "department_std", "encounter_type_std")
)

billing_with_dept = (
    billing_silver_df
    .join(enc_dept, "encounter_id", "left")
)

dept_ar_window = Window.partitionBy("department_std")

billing_windowed = billing_with_dept.withColumn(
    "dept_total_outstanding",
    F.coalesce(
        F.sum("outstanding_valid").over(dept_ar_window),
        F.lit(0.0)
    )
)
#display(billing_windowed.limit(10))

In [0]:
# ── BR-03 Step 2: Insurance and patient lookups ───────────────────────────────
ins_lookup = (
    insurance_silver_df
    .select(
        F.col("insurance_id"),
        F.col("insurer_name").alias("insurer_name"),
        F.col("coverage_type_std").alias("coverage_type")
    )
)

pat_lookup_bill = (
    patients_silver_df
    .select(
        F.col("patient_id"),
        F.col("age_group").alias("patient_age_group")
    )
)

gold_billing_summary = (
    billing_windowed
    .join(ins_lookup,      "insurance_id", "left")
    .join(pat_lookup_bill, "patient_id",   "left")
    .withColumn("insurer_name",
        F.coalesce(F.col("insurer_name"),      F.lit("Self-Pay")))
    .withColumn("coverage_type",
        F.coalesce(F.col("coverage_type"),     F.lit("Unknown")))
    .withColumn("patient_age_group",
        F.coalesce(F.col("patient_age_group"), F.lit("Unknown")))
    .withColumn("encounter_type",
        F.coalesce(F.col("encounter_type_std"),F.lit("Unknown")))
    .withColumn("out_of_pocket",
        F.coalesce(F.col("oop_amount_valid"),  F.lit(0.0)))
    .withColumn("outstanding_balance",
        F.coalesce(F.col("outstanding_valid"), F.lit(0.0)))
    .select(
        "billing_id", "encounter_id", "patient_id",
        "bill_dt", "bill_year_month",
        "gross_amount_valid", "discount_amount_valid", "net_payable",
        "ins_approved_valid", "patient_liability",
        "out_of_pocket", "outstanding_balance",
        "payment_lag_days", "ar_age_bucket",
        "bill_status_std", "payment_mode_std", "is_fully_paid",
        "insurer_name", "coverage_type",
        "patient_age_group", "encounter_type",
        "dept_total_outstanding", "dq_billing_flag"
    )
)

print(f"[BR-03] gold_billing_summary row count : {gold_billing_summary.count():,}")
display(gold_billing_summary.limit(10))

In [0]:
# ── BR-03 Write gold_billing_summary ─────────────────────────────────────────
(
    gold_billing_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("bill_year_month")
    .save(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/Gold_Billing_Summary')
)
print("[BR-03] gold_billing_summary written successfully.")

---
###  BR-04 — Clinical Quality — Diagnosis Burden
**Gold Table:** `gold_diagnosis_summary`  
**Grain:** 1 row per `patient_id` per `encounter_year`  
**Sources:** `diagnoses_silver_df`, `patients_silver_df`, `encounters_silver_df`


In [0]:
# ── BR-04 Step 1: Attach encounter_year to diagnoses via encounter_id ─────────
enc_year = (
    encounters_silver_df
    .select("encounter_id", "encounter_year")
    .filter(F.col("encounter_year").isNotNull())
    .dropDuplicates(["encounter_id"])
)

diagnoses_with_year = (
    diagnoses_silver_df
    .join(enc_year, "encounter_id", "left")
    .filter(F.col("encounter_year").isNotNull())
)
#display(diagnoses_with_year)

In [0]:
# ── BR-04 Step 2: Mode helper – most frequent diagnosed_by_provider ──────────
# Using row_number over count window to pick most frequent value (MODE)
prov_freq = (
    diagnoses_with_year
    .filter(F.col("diagnosed_by_provider").isNotNull())
    .groupBy("patient_id", "encounter_year", "diagnosed_by_provider")
    .agg(F.count("*").alias("freq"))
)

prov_window = Window.partitionBy("patient_id", "encounter_year").orderBy(
    F.col("freq").desc()
)

primary_provider = (
    prov_freq
    .withColumn("rn", F.row_number().over(prov_window))
    .filter(F.col("rn") == 1)
    .select("patient_id", "encounter_year",
            F.col("diagnosed_by_provider").alias("primary_provider_id"))
)
#display(primary_provider.limit(10))

In [0]:
# ── BR-04 Step 3: Main aggregations by patient + year ────────────────────────
dx_summary = (
    diagnoses_with_year
    .groupBy("patient_id", "encounter_year")
    .agg(
        F.count("diagnosis_id")
         .alias("total_diagnoses"),
        F.sum(F.coalesce(F.col("chronic_flag_std"), F.lit(0)))
         .alias("chronic_diagnosis_count"),
        F.round(F.avg("severity_score"), 2)
         .alias("avg_severity_score"),
        F.max("severity_score")
         .alias("max_severity_score"),
        F.max(F.coalesce(F.col("is_mental_health_dx"), F.lit(0)))
         .alias("has_mental_health_dx"),
        F.collect_list(
            F.when(F.col("diagnosis_type_std") == "Primary",
                   F.col("icd10_code_std"))
        ).alias("primary_diagnosis_codes"),
        F.collect_set("icd10_chapter")
         .alias("icd10_chapters_seen"),
        F.round(
            F.avg(
                F.when(F.col("resolution_date_std").isNotNull(),
                       F.col("diagnosis_duration_days"))
            ), 1
        ).alias("avg_diagnosis_duration_days"),
        F.count(
            F.when(F.col("diagnosis_status_std") == "Active",
                   F.col("diagnosis_id"))
        ).alias("active_diagnosis_count"),
        F.count(
            F.when(F.col("confidence_std") == "confirmed",
                   F.col("diagnosis_id"))
        ).alias("confirmed_diagnosis_count"),
        F.sum(F.coalesce(F.col("dq_dx_flag"), F.lit(0)))
         .alias("dq_issues_count")
    )
)
display(dx_summary.limit(10))

In [0]:
# ── BR-04 Step 4: Join patient name/age and primary_provider ─────────────────
pat_lookup_dx = (
    patients_silver_df
    .select(
        "patient_id",
        F.col("full_name").alias("patient_name"),
        F.col("age_group")
    )
)

gold_diagnosis_summary = (
    dx_summary
    .join(pat_lookup_dx,   "patient_id",               "left")
    .join(primary_provider,["patient_id","encounter_year"], "left")
    .withColumn("age_group",
        F.coalesce(F.col("age_group"), F.lit("Unknown")))
    .withColumn("has_mental_health_dx",
        F.coalesce(F.col("has_mental_health_dx"), F.lit(0)))
    .withColumn("chronic_diagnosis_count",
        F.coalesce(F.col("chronic_diagnosis_count"), F.lit(0)))
    .withColumn("total_diagnoses",
        F.coalesce(F.col("total_diagnoses"), F.lit(0)))
    .withColumn("dq_issues_count",
        F.coalesce(F.col("dq_issues_count"), F.lit(0)))
    .select(
        "patient_id", "encounter_year",
        "patient_name", "age_group",
        "total_diagnoses", "chronic_diagnosis_count",
        "avg_severity_score", "max_severity_score",
        "has_mental_health_dx",
        "primary_diagnosis_codes", "icd10_chapters_seen",
        "avg_diagnosis_duration_days",
        "active_diagnosis_count", "confirmed_diagnosis_count",
        "dq_issues_count", "primary_provider_id"
    )
)

print(f"[BR-04] gold_diagnosis_summary row count : {gold_diagnosis_summary.count():,}")
display(gold_diagnosis_summary.limit(10))

In [0]:
# ── BR-04 Write gold_diagnosis_summary ───────────────────────────────────────
(
    gold_diagnosis_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("encounter_year")
    .save(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/Gold_Diagnosis_Summary')
)
print("[BR-04] gold_diagnosis_summary written successfully.")

---
###  BR-05 — Medication Adherence & Prescribing Patterns
**Gold Table:** `gold_medication_insights`  
**Grain:** 1 row per `prescriber_id` per `drug_display_name`  
**Sources:** `medications_silver_df`, `providers_silver_df`, `patients_silver_df`


In [0]:
# ── BR-05 Step 1: Polypharmacy sub-query – patients on 5+ unique drugs ────────
polypharmacy_patients = (
    medications_silver_df
    .groupBy("patient_id")
    .agg(F.countDistinct("drug_display_name").alias("unique_drug_count"))
    .filter(F.col("unique_drug_count") > 5)
    .select("patient_id")
)

# Flag polypharmacy at medication level
medications_with_poly = (
    medications_silver_df
    .join(
        polypharmacy_patients.withColumn("is_polypharmacy", F.lit(1)),
        "patient_id", "left"
    )
    .withColumn("is_polypharmacy",
        F.coalesce(F.col("is_polypharmacy"), F.lit(0)))
)
#display(medications_with_poly.limit(10))

In [0]:
# ── BR-05 Step 2: Mode helpers – most common frequency & route per group ──────
freq_window  = Window.partitionBy("prescriber_id_valid","drug_display_name")
route_window = Window.partitionBy("prescriber_id_valid","drug_display_name")

freq_ranked = (
    medications_with_poly
    .filter(F.col("frequency_std").isNotNull())
    .groupBy("prescriber_id_valid","drug_display_name","frequency_std")
    .agg(F.count("*").alias("freq_cnt"))
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("prescriber_id_valid","drug_display_name")
              .orderBy(F.col("freq_cnt").desc())
    ))
    .filter(F.col("rn") == 1)
    .select(
        "prescriber_id_valid","drug_display_name",
        F.col("frequency_std").alias("most_common_frequency")
    )
)

route_ranked = (
    medications_with_poly
    .filter(F.col("route_std").isNotNull())
    .groupBy("prescriber_id_valid","drug_display_name","route_std")
    .agg(F.count("*").alias("route_cnt"))
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("prescriber_id_valid","drug_display_name")
              .orderBy(F.col("route_cnt").desc())
    ))
    .filter(F.col("rn") == 1)
    .select(
        "prescriber_id_valid","drug_display_name",
        F.col("route_std").alias("most_common_route")
    )
)
#display(route_ranked.limit(10))

In [0]:
# ── BR-05 Step 3: Main aggregations per prescriber per drug ──────────────────
med_agg = (
    medications_with_poly
    .filter(
        F.col("prescriber_id_valid").isNotNull() &
        F.col("drug_display_name").isNotNull()
    )
    .groupBy("prescriber_id_valid", "drug_display_name")
    .agg(
        F.count("medication_id")
         .alias("prescription_count"),
        F.countDistinct("patient_id")
         .alias("unique_patients"),
        F.round(F.avg("dose_amount_valid"), 2)
         .alias("avg_dose_amount"),
        F.round(F.avg("days_supply_valid"), 1)
         .alias("avg_days_supply"),
        F.round(F.avg("actual_days_supply"), 1)
         .alias("avg_actual_duration"),
        F.count(
            F.when(F.col("medication_status_std") == "Active",
                   F.col("medication_id"))
        ).alias("active_prescriptions"),
        F.countDistinct(
            F.when(F.col("is_polypharmacy") == 1, F.col("patient_id"))
        ).alias("polypharmacy_patient_count"),
        F.collect_set("indication_icd10_std")
         .alias("linked_icd10_codes"),
        F.sum(F.coalesce(F.col("dq_med_flag"), F.lit(0)))
         .alias("dq_issues_count")
    )
)
#display(med_agg.limit(10))

In [0]:
# ── BR-05 Step 4: Join provider info, mode columns ───────────────────────────
prov_lookup_med = (
    providers_silver_df
    .select(
        F.col("provider_id"),
        F.col("full_name").alias("prescriber_name"),
        F.col("specialty_std").alias("prescriber_specialty"),
        F.col("department_std").alias("prescriber_department")
    )
)

gold_medication_insights = (
    med_agg
    .join(
        prov_lookup_med,
        med_agg["prescriber_id_valid"] == prov_lookup_med["provider_id"],
        "left"
    )
    .join(
        freq_ranked,
        ["prescriber_id_valid","drug_display_name"], "left"
    )
    .join(
        route_ranked,
        ["prescriber_id_valid","drug_display_name"], "left"
    )
    .withColumn("prescriber_specialty",
        F.coalesce(F.col("prescriber_specialty"), F.lit("General")))
    .withColumn("prescriber_department",
        F.coalesce(F.col("prescriber_department"), F.lit("Unknown")))
    .withColumnRenamed("prescriber_id_valid", "prescriber_id")
    .select(
        "prescriber_id", "drug_display_name",
        "prescriber_name", "prescriber_specialty", "prescriber_department",
        "prescription_count", "unique_patients",
        "avg_dose_amount", "avg_days_supply", "avg_actual_duration",
        "active_prescriptions", "most_common_frequency", "most_common_route",
        "polypharmacy_patient_count", "linked_icd10_codes", "dq_issues_count"
    )
)

print(f"[BR-05] gold_medication_insights row count : {gold_medication_insights.count():,}")
display(gold_medication_insights.limit(10))

In [0]:
# ── BR-05 Write gold_medication_insights ─────────────────────────────────────
(
    gold_medication_insights
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("prescriber_specialty")
    .save(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/Gold_Medication_Insights')
    
)
print("[BR-05] gold_medication_insights written successfully.")

---
###  BR-06 — Lab Results & Abnormality Trends
**Gold Table:** `gold_lab_analytics`  
**Grain:** 1 row per `patient_id` per `test_name_std`  
**Sources:** `lab_results_silver_df`, `encounters_silver_df`, `patients_silver_df`


In [0]:
# ── BR-06 Step 1: Mode helper – canonical LOINC per patient per test ──────────
loinc_ranked = (
    lab_results_silver_df
    .filter(F.col("loinc_code_std").isNotNull())
    .groupBy("patient_id", "test_name_std", "loinc_code_std")
    .agg(F.count("*").alias("loinc_cnt"))
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("patient_id","test_name_std")
              .orderBy(F.col("loinc_cnt").desc())
    ))
    .filter(F.col("rn") == 1)
    .select("patient_id","test_name_std",
            F.col("loinc_code_std").alias("loinc_code"))
)
#display(lab_results_silver_df.limit(10))

In [0]:
# ── BR-06 Step 2: Main lab aggregations per patient per test ─────────────────
lab_agg = (
    lab_results_silver_df
    .filter(
        F.col("patient_id").isNotNull() &
        F.col("test_name_std").isNotNull()
    )
    .groupBy("patient_id", "test_name_std")
    .agg(
        F.count("lab_result_id")
         .alias("total_tests_ordered"),
        F.round(F.avg("result_num_valid"), 2)
         .alias("avg_result_value"),
        F.min("result_num_valid")
         .alias("min_result_value"),
        F.max("result_num_valid")
         .alias("max_result_value"),
        F.sum(F.coalesce(F.col("abnormal_flag_std"), F.lit(0)))
         .alias("abnormal_test_count"),
        F.sum(F.coalesce(F.col("critical_flag_std"), F.lit(0)))
         .alias("critical_test_count"),
        F.round(
            F.sum(F.coalesce(F.col("abnormal_flag_std"), F.lit(0))) /
            F.count("lab_result_id") * 100,
            1
        ).alias("abnormal_rate_pct"),
        F.round(F.avg("result_deviation_pct"), 2)
         .alias("avg_deviation_pct"),
        F.round(F.avg("tat_hours"), 1)
         .alias("avg_tat_hours"),
        F.min(F.to_date(F.col("collection_ts")))
         .alias("first_test_date"),
        F.max(F.to_date(F.col("collection_ts")))
         .alias("latest_test_date"),
        F.count(
            F.when(F.col("result_status_std") == "Final",
                   F.col("lab_result_id"))
        ).alias("final_result_count"),
        F.sum(F.coalesce(F.col("dq_lab_flag"), F.lit(0)))
         .alias("dq_issues_count")
    )
)
#display(lab_agg.limit(10))

In [0]:
# ── BR-06 Step 3: Join patient info and LOINC mode ───────────────────────────
pat_lookup_lab = (
    patients_silver_df
    .select(
        "patient_id",
        F.col("full_name").alias("patient_name"),
        F.col("age_group").alias("patient_age_group")
    )
)

gold_lab_analytics = (
    lab_agg
    .join(pat_lookup_lab, "patient_id", "left")
    .join(loinc_ranked,  ["patient_id","test_name_std"], "left")
    .withColumn("patient_age_group",
        F.coalesce(F.col("patient_age_group"), F.lit("Unknown")))
    .withColumn("abnormal_test_count",
        F.coalesce(F.col("abnormal_test_count"), F.lit(0)))
    .withColumn("critical_test_count",
        F.coalesce(F.col("critical_test_count"), F.lit(0)))
    .withColumn("dq_issues_count",
        F.coalesce(F.col("dq_issues_count"), F.lit(0)))
    .select(
        "patient_id", "test_name_std",
        "patient_name", "patient_age_group",
        "loinc_code",
        "total_tests_ordered",
        "avg_result_value", "min_result_value", "max_result_value",
        "abnormal_test_count", "critical_test_count", "abnormal_rate_pct",
        "avg_deviation_pct", "avg_tat_hours",
        "first_test_date", "latest_test_date",
        "final_result_count", "dq_issues_count"
    )
)

print(f"[BR-06] gold_lab_analytics row count : {gold_lab_analytics.count():,}")
display(gold_lab_analytics.limit(10))

In [0]:
# ── BR-06 Write gold_lab_analytics ───────────────────────────────────────────
(
    gold_lab_analytics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("test_name_std")
    .save(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/Gold_Lab_Analytics')
)
print("[BR-06] gold_lab_analytics written successfully.")

---
###  BR-07 — Provider Performance & Productivity
**Gold Table:** `gold_provider_performance`  
**Grain:** 1 row per `provider_id`  
**Sources:** `providers_silver_df`, `encounters_silver_df`, `billing_silver-Df`, `diagnoses_silver_df`


In [0]:
# ── BR-07 Step 1: Encounter aggregations per provider ────────────────────────
enc_prov_agg = (
    encounters_silver_df
    .filter(F.col("provider_id").isNotNull())
    .groupBy("provider_id")
    .agg(
        F.countDistinct("encounter_id")
         .alias("total_encounters_handled"),
        F.countDistinct("patient_id")
         .alias("unique_patients_seen"),
        F.round(F.avg("los_days"), 2)
         .alias("avg_los_days"),
        F.round(
            F.sum(F.coalesce(F.col("readmission_flag_std"), F.lit(0))) /
            F.count("*") * 100,
            2
        ).alias("readmission_rate_pct"),
        F.sum(F.coalesce(F.col("total_charges_valid"), F.lit(0.0)))
         .alias("total_charges_generated")
    )
)
#display(enc_prov_agg.limit(10))

In [0]:
# ── BR-07 Step 2: Billing aggregations per provider (via encounter join) ──────
bill_enc = (
    billing_silver_df
    .join(
        encounters_silver_df.select("encounter_id","provider_id"),
        "encounter_id", "inner"
    )
    .filter(F.col("provider_id").isNotNull())
)

bill_prov_agg = (
    bill_enc
    .groupBy("provider_id")
    .agg(
        F.sum(F.coalesce(F.col("gross_amount_valid"), F.lit(0.0)))
         .alias("total_billing_amount"),
        F.round(
            F.sum(F.coalesce(F.col("is_fully_paid"), F.lit(0))) /
            F.count("*") * 100,
            2
        ).alias("collection_rate_pct")
    )
)
#display(bill_prov_agg.limit(10))

In [0]:
# ── BR-07 Step 3: Diagnosis aggregations per provider ────────────────────────
dx_prov_agg = (
    diagnoses_silver_df
    .filter(F.col("diagnosed_by_provider").isNotNull())
    .groupBy(F.col("diagnosed_by_provider").alias("provider_id"))
    .agg(
        F.count("diagnosis_id")
         .alias("total_diagnoses_made"),
        F.sum(F.coalesce(F.col("chronic_flag_std"), F.lit(0)))
         .alias("chronic_cases_count")
    )
)
#display(dx_prov_agg.limit(10))

In [0]:
# ── BR-07 Step 4: Join all aggregations onto providers base ──────────────────
providers_base = (
    providers_silver_df
    .select(
        "provider_id", "full_name", "specialty_std", "department_std",
        "experience_band", "provider_type_std", "is_specialist",
        "consult_fee_valid", "is_active_flag", "dq_provider_flag"
    )
)

gold_provider_performance = (
    providers_base
    .join(enc_prov_agg,  "provider_id", "left")
    .join(bill_prov_agg, "provider_id", "left")
    .join(dx_prov_agg,   "provider_id", "left")
    .withColumn("specialty_std",
        F.coalesce(F.col("specialty_std"), F.lit("General")))
    .withColumn("department_std",
        F.coalesce(F.col("department_std"), F.lit("Unknown")))
    .withColumn("experience_band",
        F.coalesce(F.col("experience_band"), F.lit("Unknown")))
    .withColumn("provider_type_std",
        F.coalesce(F.col("provider_type_std"), F.lit("Unknown")))
    .withColumn("is_specialist",
        F.coalesce(F.col("is_specialist"), F.lit(0)))
    .withColumn("is_active_flag",
        F.coalesce(F.col("is_active_flag"), F.lit(1)))
    .withColumn("dq_provider_flag",
        F.coalesce(F.col("dq_provider_flag"), F.lit(0)))
    .withColumn("total_encounters_handled",
        F.coalesce(F.col("total_encounters_handled"), F.lit(0)))
    .withColumn("unique_patients_seen",
        F.coalesce(F.col("unique_patients_seen"), F.lit(0)))
    .withColumn("readmission_rate_pct",
        F.coalesce(F.col("readmission_rate_pct"), F.lit(0.0)))
    .withColumn("total_charges_generated",
        F.coalesce(F.col("total_charges_generated"), F.lit(0.0)))
    .withColumn("total_billing_amount",
        F.coalesce(F.col("total_billing_amount"), F.lit(0.0)))
    .withColumn("collection_rate_pct",
        F.coalesce(F.col("collection_rate_pct"), F.lit(0.0)))
    .withColumn("total_diagnoses_made",
        F.coalesce(F.col("total_diagnoses_made"), F.lit(0)))
    .withColumn("chronic_cases_count",
        F.coalesce(F.col("chronic_cases_count"), F.lit(0)))
    .withColumnRenamed("full_name",       "provider_name")
    .withColumnRenamed("specialty_std",   "specialty")
    .withColumnRenamed("department_std",  "department")
    .withColumnRenamed("provider_type_std","provider_type")
    .withColumnRenamed("is_active_flag",  "is_active")
    .select(
        "provider_id", "provider_name", "specialty", "department",
        "experience_band", "provider_type", "is_specialist",
        "total_encounters_handled", "unique_patients_seen",
        "avg_los_days", "readmission_rate_pct", "total_charges_generated",
        "total_billing_amount", "collection_rate_pct",
        "total_diagnoses_made", "chronic_cases_count",
        "consult_fee_valid",
        "is_active", "dq_provider_flag"
    )
)

print(f"[BR-07] gold_provider_performance row count : {gold_provider_performance.count():,}")
display(gold_provider_performance.limit(10))

In [0]:
# ── BR-07 Write gold_provider_performance ────────────────────────────────────
(
    gold_provider_performance
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("specialty")
    .save(r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/gold/Gold_Provider_Performance')
)
print("[BR-07] gold_provider_performance written successfully.")

---
###  Gold Layer Build Complete

| Gold Table | BR | Grain | Partition Key |
|---|---|---|---|
| `gold_patient_profile` | BR-01 | patient_id | — |
| `gold_encounter_analytics` | BR-02 | encounter_id | encounter_period |
| `gold_billing_summary` | BR-03 | billing_id | bill_year_month |
| `gold_diagnosis_summary` | BR-04 | patient_id + year | encounter_year |
| `gold_medication_insights` | BR-05 | prescriber_id + drug | prescriber_specialty |
| `gold_lab_analytics` | BR-06 | patient_id + test | test_name_std |
| `gold_provider_performance` | BR-07 | provider_id | specialty |


In [0]:
# ── Quick row count verification across all Gold tables (CORRECTED) ────────
gold_tables = [
    "Gold_Patient_Profile",
    "Gold_Encounter_Analytics",
    "Gold_Billing_Summary",
    "Gold_Diagnosis_Summary",
    "Gold_Medication_Insights",
    "Gold_Lab_Analytics",
    "Gold_Provider_Performance",
]

print("=" * 55)
print(f"{'Gold Table':<35} {'Row Count':>10}")
print("=" * 55)
for tbl in gold_tables:
    cnt = spark.read.format("delta").load(f"{gold_path}{tbl}").count()
    print(f"{tbl:<35} {cnt:>10,}")
print("=" * 55)